In [ ]:
# !pip install optuna
# !pip uninstall lightgbm -y

# # Устанавливаем зависимости для компиляции
# !apt update
# !apt install -y cmake build-essential libboost-dev libboost-system-dev libboost-filesystem-dev ocl-icd-opencl-dev

# # Устанавливаем LightGBM с поддержкой CUDA (GPU)
# !pip install lightgbm --no-binary lightgbm --config-settings=cmake.define.USE_CUDA=ON

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 29.5 MB/s eta 0:00:00
Found existing installation: lightgbm 4.6.0
Uninstalling lightgbm-4.6.0:
  Successfully uninstalled lightgbm-4.6.0
Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:4 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [95.6 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:8 http://security.ubuntu.com/ubuntu jammy-security/multiverse amd64 Packages [77.8 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,295 kB]
Get:10 http://security.ubuntu.com/ubuntu jammy-security/main amd64 Packages [3,955 kB]

In [ ]:
import lightgbm as lgb
print(lgb.__version__)
# Создайте небольшой тест, чтобы убедиться в использовании GPU
import numpy as np
from sklearn.datasets import make_classification

X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
train_data = lgb.Dataset(X, label=y)

params = {
    'objective': 'binary',
    'metric': 'auc',
    'verbosity': 1,
    'device': 'cuda',  # для CUDA-сборки
    'gpu_device_id': 0
}
gbm = lgb.train(params, train_data, num_boost_round=10)

4.6.0
[LightGBM] [Warning] Using sparse features with CUDA is currently not supported.
[LightGBM] [Info] Number of positive: 500, number of negative: 500
[LightGBM] [Warning] Metric auc is not implemented in cuda version. Fall back to evaluation on CPU.
[LightGBM] [Info] Total Bins 5100
[LightGBM] [Info] Number of data points in the train set: 1000, number of used features: 20
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.500000 -> initscore=0.000000
[LightGBM] [Warning] No further splits with positive gain, training stopped with 24 leaves.


In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, StratifiedGroupKFold
from sklearn.utils import resample
import optuna
from optuna.samplers import TPESampler
import warnings
from typing import Tuple, Dict, Any
warnings.filterwarnings('ignore')

In [ ]:
RANDOM_STATE = 42
N_TRIALS = 100          # увеличено с 50 до 100
CV_FOLDS = 3            # кросс-валидация внутри Optuna

In [ ]:
slices = pd.read_csv('slices_march.csv', parse_dates=['timestamp'])

# Воспроизводим стратифицированное разбиение по пассам (как в исходной работе)
pass_labels = slices.groupby('ticket_number')['y_true'].max().reset_index()
pass_labels.columns = ['ticket_number', 'has_transfer']

train_pass, temp_pass = train_test_split(
    pass_labels['ticket_number'], test_size=0.3, random_state=RANDOM_STATE,
    stratify=pass_labels['has_transfer']
)
val_pass, test_pass = train_test_split(
    temp_pass, test_size=0.5, random_state=RANDOM_STATE,
    stratify=pass_labels[pass_labels['ticket_number'].isin(temp_pass)]['has_transfer']
)

train_df = slices[slices['ticket_number'].isin(train_pass)].copy()
val_df   = slices[slices['ticket_number'].isin(val_pass)].copy()
test_df  = slices[slices['ticket_number'].isin(test_pass)].copy()

print(f"Train: {len(train_df)} срезов, Val: {len(val_df)}, Test: {len(test_df)}")

Train: 338598 срезов, Val: 71742, Test: 72159


In [ ]:
# Признаки (те же, что использовались ранее)
feature_cols = [c for c in slices.columns if c not in
                ['ticket_number', 'timestamp', 'y_true', 'is_labeled']]

X_train = train_df[feature_cols].values
y_train = train_df['y_true'].values
groups_train = train_df['ticket_number'].values

X_val = val_df[feature_cols].values
y_val = val_df['y_true'].values

X_test = test_df[feature_cols].values
y_test = test_df['y_true'].values

In [ ]:
def precision_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int = 20,
                   groups: np.ndarray = None) -> float:
    """
    Precision@k на уровне пассов: для каждого пасса берётся максимальный скор,
    затем пассы ранжируются, и считается доля реальных перепродаж в топ-k.
    """
    if groups is None:
        raise ValueError("Необходимо передать groups (ticket_number)")
    df = pd.DataFrame({'score': y_score, 'label': y_true, 'ticket': groups})
    pass_max = df.groupby('ticket').agg(max_score=('score', 'max'),
                                        has_transfer=('label', 'max')).reset_index()
    top_k = pass_max.nlargest(k, 'max_score')
    return top_k['has_transfer'].sum() / k

In [ ]:
class ElkanotoPU:
    """
    Метод Elkan & Noto (2008):
    1. Обучаем классификатор отличать P от U (метки: P=1, U=0).
    2. Оцениваем c = средняя вероятность принадлежности к P на подмножестве P.
    3. Истинная вероятность p(y=1|x) = p(s=1|x) / c, где s=1 – метка P.
    Возвращаем скорректированные вероятности.
    """
    def __init__(self, base_estimator):
        self.base_estimator = base_estimator
        self.c_ = None

    def fit(self, X: np.ndarray, y: np.ndarray, sample_weight: np.ndarray = None):
        # y содержит 0/1, где 1 – положительные (P), 0 – немаркированные (U)
        if sample_weight is None:
            sample_weight = np.ones(len(y))
        self.base_estimator.fit(X, y, sample_weight=sample_weight)
        # Оценка c на положительных примерах
        p_idx = np.where(y == 1)[0]
        prob_p = self.base_estimator.predict_proba(X[p_idx])[:, 1]
        self.c_ = np.mean(prob_p)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        prob = self.base_estimator.predict_proba(X)[:, 1]
        # Корректировка: p(y=1|x) = p(s=1|x) / c, с обрезанием до [0,1]
        corrected = np.clip(prob / max(self.c_, 1e-6), 0, 1)
        return corrected

class BaggingPU:
    """
    Bagging PU (Mordelet & Vert, 2014):
    Обучаем ансамбль классификаторов, каждый на всех P + случайной подвыборке U
    размера K * |P| (по умолчанию K=1). Итоговый скор – средняя вероятность.
    """
    def __init__(self, base_estimator, n_estimators: int = 20, K: int = 1,
                 random_state: int = 42):
        self.base_estimator = base_estimator
        self.n_estimators = n_estimators
        self.K = K
        self.random_state = random_state
        self.estimators_ = []

    def fit(self, X: np.ndarray, y: np.ndarray):
        p_idx = np.where(y == 1)[0]
        u_idx = np.where(y == 0)[0]
        X_p = X[p_idx]
        n_p = len(p_idx)
        np.random.seed(self.random_state)

        for i in range(self.n_estimators):
            # Случайная подвыборка немаркированных размера K * n_p
            n_u_sample = min(self.K * n_p, len(u_idx))
            u_sample_idx = np.random.choice(u_idx, size=n_u_sample, replace=False)
            X_u_sample = X[u_sample_idx]

            X_combined = np.vstack([X_p, X_u_sample])
            y_combined = np.hstack([np.ones(n_p), np.zeros(n_u_sample)])

            est = lgb.LGBMClassifier(**self.base_estimator.get_params())
            est.fit(X_combined, y_combined)
            self.estimators_.append(est)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        probas = [est.predict_proba(X)[:, 1] for est in self.estimators_]
        return np.mean(probas, axis=0)

class TwoStepPU:
    """
    Two-step PU (Spy technique):
    1. Добавляем spy-примеры из P в U (сохраняя их истинную метку).
    2. Обучаем классификатор P (без spy) vs U+spy.
    3. По предсказаниям на spy определяем порог надёжности.
    4. Выбираем надёжные отрицательные (RN) – те U, чей скор ниже порога.
    5. Обучаем финальный классификатор на P + RN.
    """
    def __init__(self, base_estimator, spy_rate: float = 0.15, random_state: int = 42):
        self.base_estimator = base_estimator
        self.spy_rate = spy_rate
        self.random_state = random_state
        self.final_estimator_ = None
        self.threshold_ = None

    def fit(self, X: np.ndarray, y: np.ndarray):
        p_idx = np.where(y == 1)[0]
        u_idx = np.where(y == 0)[0]
        np.random.seed(self.random_state)

        # Отбираем spy-примеры
        n_spy = max(1, int(len(p_idx) * self.spy_rate))
        spy_idx = np.random.choice(p_idx, size=n_spy, replace=False)
        p_clean_idx = np.setdiff1d(p_idx, spy_idx)

        # Формируем выборку: P_clean vs U + spy
        X_spy = X[spy_idx]
        y_spy = np.zeros(n_spy)          # spy помечаются как U (класс 0)
        X_train1 = np.vstack([X[p_clean_idx], X[u_idx], X_spy])
        y_train1 = np.hstack([np.ones(len(p_clean_idx)), np.zeros(len(u_idx)), y_spy])

        spy_detector = lgb.LGBMClassifier(**self.base_estimator.get_params())
        spy_detector.fit(X_train1, y_train1)

        # Определяем порог: минимум среди предсказаний на spy (они должны быть низкими)
        spy_probs = spy_detector.predict_proba(X_spy)[:, 1]
        self.threshold_ = np.min(spy_probs) if len(spy_probs) > 0 else 0.0

        # Надёжные отрицательные: U с вероятностью ниже порога
        u_probs = spy_detector.predict_proba(X[u_idx])[:, 1]
        rn_idx = u_idx[u_probs < self.threshold_]

        if len(rn_idx) == 0:
            # fallback: использовать все U как отрицательные (не оптимально, но устойчиво)
            rn_idx = u_idx

        # Финальный классификатор P vs RN
        X_final = np.vstack([X[p_idx], X[rn_idx]])
        y_final = np.hstack([np.ones(len(p_idx)), np.zeros(len(rn_idx))])

        self.final_estimator_ = lgb.LGBMClassifier(**self.base_estimator.get_params())
        self.final_estimator_.fit(X_final, y_final)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self.final_estimator_.predict_proba(X)[:, 1]

In [ ]:
def objective(trial, method_name: str, X_tr: np.ndarray, y_tr: np.ndarray,
              X_vl: np.ndarray, y_vl: np.ndarray) -> float:
    params = {
        'objective': 'binary',
        'metric': 'auc',
        'verbosity': -1,
        'n_estimators': trial.suggest_int('n_estimators', 50, 300),      # сужен диапазон
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 15, 127),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'subsample': trial.suggest_float('subsample', 0.6, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
        'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'device': 'cuda',
        'random_state': RANDOM_STATE
    }
    base_clf = lgb.LGBMClassifier(**params)

    if method_name == 'elkanoto':
        model = ElkanotoPU(base_clf)
        model.fit(X_tr, y_tr)
        prob = model.predict_proba(X_vl)
    elif method_name == 'bagging':
        model = BaggingPU(base_clf, n_estimators=5, random_state=RANDOM_STATE)
        model.fit(X_tr, y_tr)
        prob = model.predict_proba(X_vl)
    elif method_name == 'twostep':
        model = TwoStepPU(base_clf, spy_rate=0.15, random_state=RANDOM_STATE)
        model.fit(X_tr, y_tr)
        prob = model.predict_proba(X_vl)
    else:  # weighted
        w = np.where(y_tr == 1, 1.0, 0.05)
        base_clf.fit(X_tr, y_tr, sample_weight=w)
        prob = base_clf.predict_proba(X_vl)[:, 1]

    return roc_auc_score(y_vl, prob)

In [ ]:
def recall_at_k(y_true: np.ndarray, y_score: np.ndarray, k: int = 20,
                groups: np.ndarray = None) -> float:
    """
    Recall@k на уровне пассов: доля реальных перепродаж среди топ-k подозрительных пассов
    от общего числа перепродаж в тестовой выборке.
    """
    if groups is None:
        raise ValueError("Необходимо передать groups (ticket_number)")
    df = pd.DataFrame({'score': y_score, 'label': y_true, 'ticket': groups})
    pass_max = df.groupby('ticket').agg(
        max_score=('score', 'max'),
        has_transfer=('label', 'max')
    ).reset_index()
    top_k = pass_max.nlargest(k, 'max_score')
    total_positives = pass_max['has_transfer'].sum()
    if total_positives == 0:
        return 0.0
    return top_k['has_transfer'].sum() / total_positives

In [ ]:
methods = ['weighted', 'elkanoto', 'bagging', 'twostep']
best_params = {}
results = {}

for method in methods:
    print(f"\n===== Оптимизация для метода: {method} =====")
    sampler = TPESampler(seed=RANDOM_STATE)
    study = optuna.create_study(direction='maximize', sampler=sampler)
    study.optimize(
        lambda trial: objective(trial, method, X_train, y_train, X_val, y_val),
        n_trials=N_TRIALS,
        show_progress_bar=True
    )
    best_params[method] = study.best_params
    params['device'] = 'cuda'
    results[method] = study.best_value
    print(f"Лучший AUC на кросс-валидации: {study.best_value:.4f}")
    print(f"Лучшие параметры: {study.best_params}")

[I 2026-05-26 14:00:51,871] A new study created in memory with name: no-name-afa09770-ea8e-43a3-8dc5-a98b36525f6f



===== Оптимизация для метода: weighted =====


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-26 14:00:55,002] Trial 0 finished with value: 0.7996468550829 and parameters: {'n_estimators': 144, 'learning_rate': 0.08927180304353628, 'num_leaves': 97, 'max_depth': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893, 'min_child_samples': 62}. Best is trial 0 with value: 0.7996468550829.
[I 2026-05-26 14:01:00,448] Trial 1 finished with value: 0.8166281283358715 and parameters: {'n_estimators': 227, 'learning_rate': 0.010485387725194618, 'num_leaves': 124, 'max_depth': 7, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 4.4734294104626844e-07, 'reg_lambda': 5.472429642032198e-06, 'min_child_samples': 55}. Best is trial 1 with value: 0.8166281283358715.
[I 2026-05-26 14:01:03,324] Trial 2 finished with value: 0.8170672422952409 and parameters: {'n_estimators': 158, 'learning_rate': 0.019553708662745254, 'num_leaves': 84, 'max_depth': 3, 'subsam

[I 2026-05-26 14:04:44,471] A new study created in memory with name: no-name-4d9bcacc-2ee4-4f1d-a9f9-4d86bcd8d2d5


...

[I 2026-05-26 14:09:56,443] Trial 0 finished with value: 0.8201561227916997 and parameters: {'n_estimators': 144, 'learning_rate': 0.08927180304353628, 'num_leaves': 97, 'max_depth': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893, 'min_child_samples': 62}. Best is trial 0 with value: 0.8201561227916997.
[I 2026-05-26 14:10:03,601] Trial 1 finished with value: 0.8132864918895365 and parameters: {'n_estimators': 227, 'learning_rate': 0.010485387725194618, 'num_leaves': 124, 'max_depth': 7, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 4.4734294104626844e-07, 'reg_lambda': 5.472429642032198e-06, 'min_child_samples': 55}. Best is trial 0 with value: 0.8201561227916997.
[I 2026-05-26 14:10:13,007] Trial 2 finished with value: 0.8205063129442786 and parameters: {'n_estimators': 158, 'learning_rate': 0.019553708662745254, 'num_leaves': 84, 'max_depth': 3, '

[I 2026-05-26 14:16:47,544] A new study created in memory with name: no-name-5b1f53cf-e944-4dea-8212-252b2202b390


[I 2026-05-26 14:16:47,540] Trial 99 finished with value: 0.8181335696246161 and parameters: {'n_estimators': 134, 'learning_rate': 0.057587186377037324, 'num_leaves': 101, 'max_depth': 7, 'subsample': 0.707620311723766, 'colsample_bytree': 0.8502452456205296, 'reg_alpha': 1.4950670815360643e-06, 'reg_lambda': 0.017803652052916105, 'min_child_samples': 47}. Best is trial 72 with value: 0.8239996759870387.
Лучший AUC на кросс-валидации: 0.8240
Лучшие параметры: {'n_estimators': 127, 'learning_rate': 0.07005388798993349, 'num_leaves': 125, 'max_depth': 7, 'subsample': 0.6113582224290375, 'colsample_bytree': 0.6377227123144064, 'reg_alpha': 0.00010271324038645754, 'reg_lambda': 0.2604775818445209, 'min_child_samples': 64}

===== Оптимизация для метода: twostep =====


  0%|          | 0/100 [00:00<?, ?it/s]

[I 2026-05-26 14:16:53,176] Trial 0 finished with value: 0.7373983365799229 and parameters: {'n_estimators': 144, 'learning_rate': 0.08927180304353628, 'num_leaves': 97, 'max_depth': 6, 'subsample': 0.6624074561769746, 'colsample_bytree': 0.662397808134481, 'reg_alpha': 3.3323645788192616e-08, 'reg_lambda': 0.6245760287469893, 'min_child_samples': 62}. Best is trial 0 with value: 0.7373983365799229.
[I 2026-05-26 14:17:04,012] Trial 1 finished with value: 0.7032615376111471 and parameters: {'n_estimators': 227, 'learning_rate': 0.010485387725194618, 'num_leaves': 124, 'max_depth': 7, 'subsample': 0.6849356442713105, 'colsample_bytree': 0.6727299868828402, 'reg_alpha': 4.4734294104626844e-07, 'reg_lambda': 5.472429642032198e-06, 'min_child_samples': 55}. Best is trial 0 with value: 0.7373983365799229.
[I 2026-05-26 14:17:09,829] Trial 2 finished with value: 0.7218644485485337 and parameters: {'n_estimators': 158, 'learning_rate': 0.019553708662745254, 'num_leaves': 84, 'max_depth': 3, '

In [ ]:
print("\nФинальное обучение и оценка на тесте")
final_models = {}
test_precisions = {}
test_recalls = {}

for method in methods:
    params = best_params[method]
    params['device'] = 'cuda'
    base_clf = lgb.LGBMClassifier(**params, random_state=RANDOM_STATE)

    if method == 'weighted':
        w = np.where(y_train == 1, 1.0, 0.05)
        base_clf.fit(X_train, y_train, sample_weight=w)
        y_pred_test = base_clf.predict_proba(X_test)[:, 1]
        final_models[method] = base_clf
    elif method == 'elkanoto':
        model = ElkanotoPU(base_clf)
        model.fit(X_train, y_train)
        y_pred_test = model.predict_proba(X_test)
        final_models[method] = model
    elif method == 'bagging':
        model = BaggingPU(base_clf, n_estimators=5, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        y_pred_test = model.predict_proba(X_test)
        final_models[method] = model
    elif method == 'twostep':
        model = TwoStepPU(base_clf, spy_rate=0.15, random_state=RANDOM_STATE)
        model.fit(X_train, y_train)
        y_pred_test = model.predict_proba(X_test)
        final_models[method] = model

    # Precision@20 на уровне пассов
    prec20 = precision_at_k(y_test, y_pred_test, k=20, groups=test_df['ticket_number'].values)
    rec20 = recall_at_k(y_test, y_pred_test, k=20, groups=test_df['ticket_number'].values)
    test_precisions[method] = prec20
    test_recalls[method] = rec20
    auc = roc_auc_score(y_test, y_pred_test)
    print(f"{method}: AUC={auc:.4f}, Precision@20={prec20:.4f}, Recall@20={rec20:.4f}")


Финальное обучение и оценка на тесте
weighted: AUC=0.8416, Precision@20=0.3500, Recall@20=0.1944
elkanoto: AUC=0.8447, Precision@20=0.2500, Recall@20=0.1389
bagging: AUC=0.8444, Precision@20=0.1000, Recall@20=0.0556
twostep: AUC=0.7944, Precision@20=0.0000, Recall@20=0.0000


In [ ]:
print("\nСводная таблица")
print("Метод\t\tЛучший CV AUC\tTest Precision@20\tTest Recall@20")
for method in methods:
    print(f"{method:<10}\t{results[method]:.4f}\t\t{test_precisions[method]:.4f}\t\t{test_recalls[method]:.4f}")


Сводная таблица
Метод		Лучший CV AUC	Test Precision@20	Test Recall@20
weighted  	0.8266		0.3500		0.1944
elkanoto  	0.8278		0.2500		0.1389
bagging   	0.8240		0.1000		0.0556
twostep   	0.7902		0.0000		0.0000
